<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/04-statistical-learning-theory.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Statistical Learning Theory and Generalization**

A learning algorithm only sees a finite dataset, yet its real purpose is to make reliable predictions on examples that were not used for training. **Statistical learning theory** studies when and why that leap is justified. It connects four objects that are easy to blur together in practice: the unknown data-generating distribution, the observed sample, the family of models that the learner is allowed to choose from, and the algorithm that performs the choice.

Let $(X,Y)\sim P$ denote a future input-label pair drawn from an unknown population distribution $P$. A training sample is

$$
S=\{(x_i,y_i)\}_{i=1}^{n}\sim P^n,
$$

where $n$ is the number of examples and $P^n$ means that the examples are assumed to be independently and identically distributed from $P$. A hypothesis $h\in\mathcal H$ maps an input to a prediction, while a loss $\ell(h(x),y)$ measures the cost of that prediction. The **hypothesis space** $\mathcal H$ may be a collection of lines, trees of bounded depth, neural networks with a given architecture, or any other family the algorithm can search.

The central question is not merely, "Can the model fit $S$?" It is:

> Under what assumptions does performance measured on $S$ provide evidence about performance on a fresh draw from $P$?

The usual theory assumes that training and future examples share the same distribution and that the loss is well defined and sufficiently controlled. Time dependence, user-level correlation, feedback loops, label leakage, or deployment shift can invalidate those assumptions even when a mathematical bound is correct. Statistical learning theory is therefore a language for stating guarantees and their conditions, not a certificate that any dataset is representative.

### **Population Risk and Empirical Risk**

#### **Expected Risk**

The quantity that ultimately matters is **population risk**, also called expected risk or true risk:

$$
R(h)=\mathbb E_{(X,Y)\sim P}\left[\ell(h(X),Y)\right].
$$

Every symbol has a distinct role:

| Symbol | Meaning |
|---|---|
| $P$ | the unknown distribution of future examples |
| $(X,Y)$ | a random input-label pair drawn from $P$ |
| $h$ | one prediction rule in $\mathcal H$ |
| $\ell$ | a task-appropriate loss, such as squared error or log loss |
| $R(h)$ | average loss over indefinitely many future draws |

For regression with squared loss, $\ell(h(x),y)=(h(x)-y)^2$, so $R(h)$ is the expected squared prediction error. For binary classification with zero-one loss, $\ell(h(x),y)=\mathbf 1[h(x)\neq y]$, so $R(h)$ is the population misclassification probability. The loss determines what "good" means; the same predictor can have different risks under different losses.

Population risk is normally unobservable because $P$ is unknown. Even a large held-out test set gives an estimate of $R(h)$, not the expectation itself. This distinction matters whenever two models have nearly equal test scores: finite-sample uncertainty may be larger than their reported difference.

![Population risk is defined over an unknown population, while ERM and SRM operate on a finite sample and a chosen hypothesis space.](assets/risk-erm-srm.svg){fig-align="center" width="100%" fig-alt="Diagram from an unknown population through an iid sample and a hypothesis space to empirical and structural risk minimization"}

*Diagram synthesized from the learning-theory setup in [Stanford CS229 notes](https://cs229.stanford.edu/notes_archive/cs229-notes4.pdf) and [Cornell CS4780 statistical learning theory](https://www.cs.cornell.edu/courses/cs4780/2019fa/lectures/18-slt2.pdf).*

Because $R(h)$ cannot be calculated directly, learning uses the **empirical risk**

$$
\widehat R_S(h)=\frac{1}{n}\sum_{i=1}^{n}\ell(h(x_i),y_i).
$$

For a *fixed* $h$, this is a sample average estimating an expectation. Its randomness comes from $S$. If the examples are independent, the loss has finite variance, and the sample is representative, $\widehat R_S(h)$ tends to approach $R(h)$ as $n$ grows. The difference

$$
R(h)-\widehat R_S(h)
$$

is a **generalization gap**. Its sign is not guaranteed for one fixed model and one sample, although a model selected for unusually low training error often has an optimistic empirical risk.

<details>
<summary><strong>Python example: empirical risk fluctuates around population risk</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(4)

def data_process(size, generator):
    x = generator.uniform(-1.0, 1.0, size=size)
    y = np.sin(np.pi * x) + generator.normal(scale=0.25, size=size)
    return x, y

def fixed_predictor(x):
    # The hypothesis is fixed before any sample is inspected.
    return 0.8 * x

def mean_squared_risk(x, y):
    return np.mean((fixed_predictor(x) - y) ** 2)

# A very large Monte Carlo sample is used only as a population-risk proxy.
x_population, y_population = data_process(300_000, rng)
population_risk = mean_squared_risk(x_population, y_population)

empirical_risks = []
for _ in range(10):
    x_sample, y_sample = data_process(30, rng)
    empirical_risks.append(mean_squared_risk(x_sample, y_sample))

print("population-risk proxy:", round(population_risk, 4))
print("ten empirical risks:   ", np.round(empirical_risks, 4))
print("empirical mean/std:    ", round(np.mean(empirical_risks), 4), round(np.std(empirical_risks), 4))
```

</details>

The experiment separates two sources of error. The fixed linear rule is systematically unable to represent the sine function, while each 30-example estimate also fluctuates because it contains different inputs and noise. More data reduces the second source but does not repair the first.

#### **Empirical Risk Minimization**

**Empirical risk minimization (ERM)** chooses a hypothesis with the lowest measured training loss:

$$
\widehat h_S\in\arg\min_{h\in\mathcal H}\widehat R_S(h).
$$

The subscript $S$ is important: a different sample can produce a different minimizer. ERM is a general principle rather than one specific optimizer. Least-squares regression, maximum-likelihood estimation through negative log-likelihood, and a fully grown decision tree can all be viewed as ERM procedures under different hypothesis spaces and losses.

ERM creates a subtle selection effect. The same data are used both to compare candidates and to report the winner's training error. If $\mathcal H$ is very flexible, some hypothesis may fit accidental sample noise. Its low $\widehat R_S$ is real for the observed sample but need not imply low $R$. This is why a concentration statement for one fixed hypothesis is insufficient: the selected $\widehat h_S$ depends on the data.

<details>
<summary><strong>Python example: ERM can prefer a model with a larger population risk</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(12)

def true_function(x):
    return np.sin(np.pi * x)

x_train = rng.uniform(-1.0, 1.0, size=35)
y_train = true_function(x_train) + rng.normal(scale=0.25, size=len(x_train))

# A large fresh sample approximates future population risk under the same process.
x_future = rng.uniform(-1.0, 1.0, size=20_000)
y_future = true_function(x_future) + rng.normal(scale=0.25, size=len(x_future))

rows = []
for degree in [1, 3, 5, 15]:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression(),
    )
    model.fit(x_train[:, None], y_train)
    train_mse = mean_squared_error(y_train, model.predict(x_train[:, None]))
    future_mse = mean_squared_error(y_future, model.predict(x_future[:, None]))
    rows.append((degree, train_mse, future_mse, future_mse - train_mse))

print("degree | train MSE | future MSE | gap")
for degree, train_mse, future_mse, gap in rows:
    print(f"{degree:6d} | {train_mse:9.4f} | {future_mse:10.4f} | {gap:7.4f}")
```

</details>

The highest-degree model can drive training error down by bending around individual observations. ERM has not "failed" mathematically: it solved the objective it was given. The failure is treating the empirical objective and the chosen hypothesis space as sufficient proxies for deployment risk.

#### **Structural Risk Minimization**

**Structural risk minimization (SRM)** organizes hypotheses into increasingly expressive classes

$$
\mathcal H_1\subset\mathcal H_2\subset\cdots
$$

and balances fit against capacity. A practical abstraction is

$$
\widehat h
=\arg\min_{h\in\mathcal H}
\left[\widehat R_S(h)+\lambda\,\Omega(h)\right],
$$

where $\Omega(h)$ measures complexity and $\lambda\geq0$ controls how much complexity is penalized. Classical SRM derives penalties from capacity bounds. Modern practice often implements the same principle through regularization, depth limits, margin control, architecture choice, early stopping, or validation-based hyperparameter selection.

The penalty must correspond to a meaningful notion of capacity. Penalizing polynomial degree can be sensible in a toy polynomial family; penalizing raw parameter count may be misleading for weight-shared or heavily regularized neural networks. The value of $\lambda$ is also not supplied by theory automatically in most real workflows and should be chosen without using the final test set.

<details>
<summary><strong>Python example: a toy structural-risk criterion balances fit and degree</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(21)
true_function = lambda x: np.sin(np.pi * x)

x_train = rng.uniform(-1.0, 1.0, 55)
y_train = true_function(x_train) + rng.normal(scale=0.30, size=len(x_train))
x_test = rng.uniform(-1.0, 1.0, 10_000)
y_test = true_function(x_test) + rng.normal(scale=0.30, size=len(x_test))

penalty_strength = 0.008
records = []
for degree in range(1, 16):
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression(),
    )
    model.fit(x_train[:, None], y_train)
    train_mse = mean_squared_error(y_train, model.predict(x_train[:, None]))
    criterion = train_mse + penalty_strength * degree
    test_mse = mean_squared_error(y_test, model.predict(x_test[:, None]))
    records.append((degree, train_mse, criterion, test_mse))

erm = min(records, key=lambda row: row[1])
srm = min(records, key=lambda row: row[2])
oracle = min(records, key=lambda row: row[3])  # visible only in this simulation

print("selection       degree  train_MSE  criterion  future_MSE")
for label, row in [("ERM", erm), ("toy SRM", srm), ("future oracle", oracle)]:
    print(f"{label:13s} {row[0]:6d} {row[1]:10.4f} {row[2]:10.4f} {row[3]:11.4f}")
```

</details>

The simulated oracle is not available in a real project because it uses future labels. It is printed only to show what the criterion is trying to approximate. A complexity penalty does not guarantee the oracle model, but it reduces the incentive to purchase a tiny training improvement with a large increase in flexibility.


### **Hypothesis Spaces and Inductive Bias**

A learner cannot infer a unique rule from finite observations without assumptions. Many functions agree on every training point and disagree elsewhere. The restrictions that make one continuation preferable to another form the learner's **inductive bias**.

The hypothesis space supplies one part of that bias. Linear regression assumes the response is well approximated by a linear combination of chosen features. A shallow tree prefers a short hierarchy of axis-aligned rules. A convolutional network encodes locality and weight sharing. Regularization, optimization, preprocessing, and data augmentation add further preferences even when the nominal architecture stays fixed.

Inductive bias is unavoidable and not synonymous with statistical bias. A useful bias captures stable structure in the problem; a harmful one excludes the relationship needed at deployment. The practical question is whether the assumptions encoded by $\mathcal H$ and the algorithm match the domain.

#### **Model Capacity**

**Capacity** describes how many distinct input-output relationships a model family can represent. A higher-capacity class can fit a wider range of datasets. Capacity may be controlled by polynomial degree, tree depth and leaf count, the number and norm of linear features, a kernel bandwidth, a margin, a neural architecture, or constraints on parameters.

Parameter count is only a rough proxy. Two models with the same number of parameters can behave differently because their parameterization, norms, margins, invariances, sparsity, optimizer, and input distribution differ. Conversely, an overparameterized model can interpolate training data while its algorithm consistently selects a comparatively simple solution under another complexity measure.

Capacity creates two opposing possibilities:

- too little capacity prevents the family from representing the underlying relationship;
- too much uncontrolled capacity makes finite-sample accidents easier to fit.

More data can support a larger class because it provides more evidence for distinguishing genuine structure from accidental fit. Capacity should therefore be discussed relative to sample size, noise, feature geometry, and the learning algorithm rather than as an absolute label such as "complex model."

#### **Approximation and Estimation Error**

Let $h^*$ be a population-risk minimizer over all predictors under consideration, and let

$$
h^*_{\mathcal H}\in\arg\min_{h\in\mathcal H}R(h)
$$

be the best predictor available inside $\mathcal H$. If $\widehat h$ is learned from a finite sample, then its excess risk can be decomposed as

$$
R(\widehat h)-R(h^*)
=\underbrace{R(h^*_{\mathcal H})-R(h^*)}_{\text{approximation error}}
+\underbrace{R(\widehat h)-R(h^*_{\mathcal H})}_{\text{estimation error}}.
$$

**Approximation error** is caused by the restrictions of $\mathcal H$. Even with infinite data and perfect optimization, a straight line cannot reproduce an arbitrary oscillating function. **Estimation error** is caused by choosing from $\mathcal H$ using only a finite random sample. A more expressive class often reduces approximation error but can increase estimation uncertainty.

![Excess risk separates the inability of a hypothesis space to express the ideal predictor from the finite-sample error of selecting within that space.](assets/capacity-error-decomposition.svg){fig-align="center" width="100%" fig-alt="Nested predictor spaces showing approximation, estimation, and optimization error"}

If an algorithm does not find the intended empirical minimizer, **optimization error** adds another gap. These three errors answer different questions:

| Error | Diagnostic question | Typical response |
|---|---|---|
| Approximation | Can this family represent the pattern? | richer features or model class |
| Estimation | Does the sample identify the right member reliably? | more representative data, regularization, lower capacity |
| Optimization | Did training reach a suitable solution? | optimizer, learning rate, initialization, numerical checks |

The terms are conceptually useful but not always directly measurable because $P$, $h^*$, and $h^*_{\mathcal H}$ are unknown.

<details>
<summary><strong>Python example: estimate approximation and estimation effects in a controlled simulation</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(31)
grid = np.linspace(-1.0, 1.0, 1_000)
truth = np.sin(np.pi * grid)

def fit_polynomial(x, y, degree):
    # np.polynomial uses increasing powers and is stable on the scaled interval [-1, 1].
    return np.polynomial.polynomial.polyfit(x, y, deg=degree)

def predict_polynomial(coefficients, x):
    return np.polynomial.polynomial.polyval(x, coefficients)

print("degree | approximation | estimation | total-to-truth")
for degree in [1, 3, 7]:
    # A dense noiseless grid approximates the best-in-class population predictor.
    population_coefficients = fit_polynomial(grid, truth, degree)
    best_in_class = predict_polynomial(population_coefficients, grid)
    approximation = np.mean((best_in_class - truth) ** 2)

    learned_predictions = []
    for _ in range(150):
        x_sample = rng.uniform(-1.0, 1.0, 30)
        y_sample = np.sin(np.pi * x_sample) + rng.normal(scale=0.25, size=30)
        sample_coefficients = fit_polynomial(x_sample, y_sample, degree)
        learned_predictions.append(predict_polynomial(sample_coefficients, grid))

    learned_predictions = np.asarray(learned_predictions)
    estimation = np.mean((learned_predictions - best_in_class) ** 2)
    total_to_truth = np.mean((learned_predictions - truth) ** 2)
    print(f"{degree:6d} | {approximation:13.5f} | {estimation:10.5f} | {total_to_truth:14.5f}")
```

</details>

The degree-1 family has a large representation limitation. Degree 3 captures the main shape with moderate estimation uncertainty. Degree 7 has almost no approximation error on the dense grid, but fitting its interacting coefficients from only 30 noisy observations is unstable. The total is not exactly the sum of the two displayed Monte Carlo quantities because finite simulations can retain a cross term; the experiment is a diagnostic illustration, not a direct estimator of an unknown real-world decomposition.


### **Underfitting and Overfitting**

**Underfitting** occurs when the learned system cannot achieve sufficiently low error even on the training distribution because its representation, features, optimization, or training time are inadequate. **Overfitting** occurs when performance on the training sample is substantially better than performance on unseen examples from the intended distribution. The first is associated with insufficient effective capacity; the second with an unreliable generalization gap.

These are empirical diagnoses, not permanent properties of a model name. A depth-10 tree can underfit a highly structured problem and overfit a tiny noisy one. A large neural network can generalize well under strong data, optimization, and regularization biases. Always specify the dataset, loss, split, and training procedure.

![Underfit, fit, and overfit models differ in how training-set performance transfers to real-world data.](assets/underfit-fit-overfit.svg){fig-align="center" width="72%" fig-alt="Google teaching curve relating training prediction quality to real-world prediction quality for underfit, fit, and overfit models"}

*Source: [Google Machine Learning Crash Course, Overfitting](https://developers.google.com/machine-learning/crash-course/overfitting/overfitting).*

#### **Training, Validation, and Generalization Error**

The three standard data roles must remain distinct:

- the **training set** estimates gradients or model parameters;
- the **validation set** compares hyperparameters, features, thresholds, and stopping times;
- the **test set** estimates performance once after the development choices are fixed.

The validation set is part of the learning process. Repeatedly trying alternatives and selecting the best validation score adapts decisions to validation noise, making that score optimistic. A final untouched test set, nested cross-validation, or an external evaluation set is needed for an honest generalization estimate. Chapter 06 develops the experimental protocol in detail.

Training error is normally lower because the model was chosen using those observations. The pattern of errors is more informative than either number alone:

| Training error | Validation error | Likely interpretation |
|---|---|---|
| high | high and similar | underfitting, weak features, optimization failure, or noisy target |
| low | much higher | overfitting, leakage in training, or distribution mismatch |
| low | low and similar | promising fit, subject to test uncertainty and deployment shift |
| high | unexpectedly lower | split mismatch, small-sample noise, or a pipeline bug |

<details>
<summary><strong>Python example: build a validation curve over polynomial capacity</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(44)
true_function = lambda x: np.cos(1.5 * np.pi * x)

x_train = rng.uniform(0.0, 1.0, 45)
y_train = true_function(x_train) + rng.normal(scale=0.15, size=len(x_train))
x_validation = rng.uniform(0.0, 1.0, 2_000)
y_validation = true_function(x_validation) + rng.normal(scale=0.15, size=len(x_validation))

print("degree | training MSE | validation MSE | generalization gap")
for degree in [1, 2, 4, 8, 12, 16]:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression(),
    )
    model.fit(x_train[:, None], y_train)
    train_mse = mean_squared_error(y_train, model.predict(x_train[:, None]))
    validation_mse = mean_squared_error(
        y_validation, model.predict(x_validation[:, None])
    )
    print(
        f"{degree:6d} | {train_mse:12.4f} | {validation_mse:14.4f} |"
        f" {validation_mse - train_mse:18.4f}"
    )
```

</details>

A **validation curve** varies a capacity-controlling hyperparameter while keeping the available training size fixed. It helps locate high-bias and high-variance regions, but selecting the best point still consumes validation information.

#### **Learning Curves**

A **learning curve** varies the number of training examples and measures both training and validation performance. This is different from a loss-versus-iteration curve: its horizontal axis is data quantity, not optimization time.

Typical patterns are:

- both curves plateau at poor performance with a small gap: more of the same data may help little until representation or features improve;
- training performance is strong but validation lags and the gap narrows with sample size: more representative data may reduce variance;
- both curves continue improving: more data is plausibly valuable;
- curves remain separated or unstable: capacity, leakage, non-i.i.d. splits, or distribution shift needs investigation.

![Learning curves show how training and cross-validation scores change as the number of training examples increases.](assets/learning-curves.png){fig-align="center" width="92%" fig-alt="Scikit-learn learning curves for Gaussian naive Bayes and an SVC as training size grows"}

*Source: [scikit-learn, Plotting Learning Curves and Checking Models' Scalability](https://scikit-learn.org/stable/auto_examples/model_selection/plot_learning_curve.html). In that figure, “test score” means the held-out cross-validation score, not a repeatedly consulted final test set.*

<details>
<summary><strong>Python example: calculate a learning curve with cross-validation</strong></summary>

```python
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(52)
x = rng.uniform(-1.0, 1.0, 320)
y = np.sin(np.pi * x) + rng.normal(scale=0.30, size=len(x))

model = make_pipeline(
    PolynomialFeatures(degree=8, include_bias=False),
    StandardScaler(),
    Ridge(alpha=0.2),
)

sizes, train_scores, validation_scores = learning_curve(
    model,
    x[:, None],
    y,
    train_sizes=[30, 60, 120, 200, 256],
    cv=5,
    scoring="neg_mean_squared_error",
    shuffle=True,
    random_state=52,
)

print("samples | train MSE | validation MSE | gap")
for size, train_score, validation_score in zip(sizes, train_scores, validation_scores):
    train_mse = -train_score.mean()
    validation_mse = -validation_score.mean()
    print(f"{size:7d} | {train_mse:9.4f} | {validation_mse:14.4f} | {validation_mse-train_mse:7.4f}")
```

</details>

Learning curves are diagnostic rather than causal proof. A narrowing gap suggests a variance-related problem, but merely collecting more examples from the wrong distribution will not fix mismatch, label bias, leakage, or an unsuitable objective.


### **Bias-Variance Decomposition**

Bias and variance explain how a fitted predictor changes across hypothetical training datasets. They are properties of a complete learning procedure: hypothesis space, optimizer, regularization, sample size, and data distribution together.

#### **Bias, Variance, and Irreducible Noise**

Assume a regression process

$$
Y=f(X)+\varepsilon,
\qquad \mathbb E[\varepsilon\mid X]=0,
\qquad \operatorname{Var}(\varepsilon\mid X=x)=\sigma^2,
$$

and let $\widehat f_S(x)$ be the prediction obtained after training on random sample $S$. For squared loss at a fixed input $x$,

$$
\mathbb E_{S,\varepsilon}\left[(Y-\widehat f_S(x))^2\mid X=x\right]
=\underbrace{\left(\mathbb E_S[\widehat f_S(x)]-f(x)\right)^2}_{\text{bias}^2}
+\underbrace{\mathbb E_S\left[(\widehat f_S(x)-\mathbb E_S\widehat f_S(x))^2\right]}_{\text{variance}}
+\underbrace{\sigma^2}_{\text{irreducible noise}}.
$$

The **bias** term measures systematic displacement of the learner's average prediction from the regression function. **Variance** measures sensitivity to which training sample was observed. **Irreducible noise** is variation in $Y$ that cannot be predicted from the available $X$ under the assumed process.

The formula requires squared error, the stated conditional mean assumption, and averaging over repeated datasets. It is not a universal identity that can be copied unchanged to accuracy, F1, or arbitrary classification loss. Related decompositions exist, but their terms and interpretations differ.

<details>
<summary><strong>Python example: verify the squared-error bias-variance decomposition</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(64)
noise_sd = 0.25
grid = np.linspace(-1.0, 1.0, 160)
true_values = np.sin(np.pi * grid)

print("degree | bias^2 | variance | noise | sum | simulated test MSE")
for degree in [1, 3, 7]:
    predictions = []
    for _ in range(220):
        x_train = rng.uniform(-1.0, 1.0, 28)
        y_train = np.sin(np.pi * x_train) + rng.normal(scale=noise_sd, size=28)
        model = make_pipeline(
            PolynomialFeatures(degree=degree, include_bias=False),
            LinearRegression(),
        )
        model.fit(x_train[:, None], y_train)
        predictions.append(model.predict(grid[:, None]))

    predictions = np.asarray(predictions)
    mean_prediction = predictions.mean(axis=0)
    bias_squared = np.mean((mean_prediction - true_values) ** 2)
    variance = np.mean(np.var(predictions, axis=0))
    noise = noise_sd ** 2

    # Give every learned model an independent noisy future target at each x.
    future_targets = true_values + rng.normal(scale=noise_sd, size=predictions.shape)
    simulated_mse = np.mean((predictions - future_targets) ** 2)
    decomposition = bias_squared + variance + noise
    print(
        f"{degree:6d} | {bias_squared:6.4f} | {variance:8.4f} |"
        f" {noise:5.4f} | {decomposition:6.4f} | {simulated_mse:18.4f}"
    )
```

</details>

Monte Carlo noise prevents exact equality, but the simulated test error should be close to the sum. The experiment also shows that a model can have small average bias and still be unreliable because its predictions vary dramatically across samples.

#### **The Bias-Variance Trade-Off**

Increasing effective capacity usually lowers bias because the learner can represent more patterns. It may raise variance because more rules agree with a small sample. Regularization, ensembling, and more data often reduce variance; richer features or model classes often reduce bias. These are tendencies rather than laws.

| Intervention | Typical effect | Important caveat |
|---|---|---|
| increase model capacity | lower bias, potentially higher variance | optimization and implicit bias can change both |
| strengthen regularization | lower variance, potentially higher bias | the right penalty depends on representation |
| add representative data | often lowers variance | does not repair systematic mismatch or label bias |
| average diverse models | lowers variance | limited when component errors are highly correlated |
| improve informative features | can lower bias without large variance cost | feature selection can leak validation information |

The classical U-shaped validation curve summarizes one common regime, but bias and variance themselves do not require test risk to be U-shaped as a single scalar notion of capacity increases. Modern interpolation and double descent, discussed later, show why the simple cartoon is not universal.


### **Regularization and Capacity Control**

Regularization changes the learning procedure so that fitting every training detail is no longer the only priority. It encodes a preference among hypotheses that may have similar empirical risk.

#### **Explicit and Implicit Regularization**

**Explicit regularization** adds a visible term or constraint, for example

$$
\min_w\left[\widehat R_S(w)+\lambda\Omega(w)\right]
\qquad\text{or}\qquad
\min_w\widehat R_S(w)\ \text{subject to}\ \Omega(w)\leq c.
$$

$\Omega(w)$ defines which parameters count as complex, while $\lambda$ or $c$ sets the strength. These constrained and penalized forms are closely related under suitable convex conditions, but their numerical hyperparameters are not interchangeable across scaling conventions.

**Implicit regularization** arises without an explicit penalty. Gradient descent may prefer particular minimum-norm or large-margin solutions; early stopping limits how completely high-variance directions are fitted; architecture imposes weight sharing or locality; stochastic optimization and finite precision can favor some interpolating solutions over others. Calling a mechanism implicit does not mean it is mysterious or always beneficial: it means the preference is induced by the procedure rather than written as a separate $\Omega$ term.

#### **L1 and L2 Penalties**

For $w\in\mathbb R^d$, the common penalties are

$$
\Omega_{L1}(w)=\|w\|_1=\sum_{j=1}^{d}|w_j|,
\qquad
\Omega_{L2}(w)=\|w\|_2^2=\sum_{j=1}^{d}w_j^2.
$$

L2 regularization shrinks coefficients smoothly and tends to share weight among correlated features. L1 regularization has a non-differentiable corner at zero and often produces sparse coefficient vectors. Feature scale matters: penalizing a coefficient attached to a large-scale feature is not comparable with penalizing one attached to a small-scale feature, so standardization is usually required.

Neither penalty is automatically a feature-selection truth detector. With correlated predictors, L1 may choose one member unstablely; L2 can retain many weak coefficients. The penalty should reflect the desired inductive bias and be tuned inside the validation protocol.

<details>
<summary><strong>Python example: compare unregularized, L2, and L1 linear models</strong></summary>

```python
import numpy as np
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(73)
n_train, n_test, n_features = 140, 4_000, 24

# Correlated features arise from five shared latent factors.
latent_train = rng.normal(size=(n_train, 5))
latent_test = rng.normal(size=(n_test, 5))
mixing = rng.normal(size=(5, n_features))
x_train = latent_train @ mixing + 0.25 * rng.normal(size=(n_train, n_features))
x_test = latent_test @ mixing + 0.25 * rng.normal(size=(n_test, n_features))

true_weights = np.zeros(n_features)
true_weights[[1, 7, 14, 20]] = [2.0, -1.5, 1.2, -0.8]
y_train = x_train @ true_weights + rng.normal(scale=1.0, size=n_train)
y_test = x_test @ true_weights + rng.normal(scale=1.0, size=n_test)

models = {
    "unregularized": LinearRegression(),
    "L2 Ridge": Ridge(alpha=0.1),
    "L1 Lasso": Lasso(alpha=0.03, max_iter=20_000),
}

print("model          test MSE  coefficient L2  non-zero coefficients")
for name, estimator in models.items():
    model = make_pipeline(StandardScaler(), estimator)
    model.fit(x_train, y_train)
    coefficients = model[-1].coef_
    test_mse = mean_squared_error(y_test, model.predict(x_test))
    non_zero = np.count_nonzero(np.abs(coefficients) > 1e-8)
    print(f"{name:14s} {test_mse:8.4f} {np.linalg.norm(coefficients):15.4f} {non_zero:22d}")
```

</details>

This single simulation does not establish that one penalty is universally superior. It reveals the different solution structures they encourage. Repeating the experiment with new samples is essential when coefficient stability matters.

#### **Early Stopping and Data Augmentation**

During iterative training, low-frequency or strongly supported structure is often fitted before weak, noisy directions. **Early stopping** monitors a validation metric and retains the parameters from the best iteration rather than the final iteration. The stopping time becomes a capacity-controlling hyperparameter and must be selected without peeking at the test set.

![Training loss can continue to decrease after validation loss has begun to rise; early stopping retains the model near the validation minimum.](assets/generalization-loss-curves.png){fig-align="center" width="92%" fig-alt="Google loss curves where training loss decreases while validation loss eventually rises"}

*Source: [Google Machine Learning Crash Course, Overfitting](https://developers.google.com/machine-learning/crash-course/overfitting/overfitting).*

<details>
<summary><strong>Python example: implement validation-based early stopping in gradient descent</strong></summary>

```python
import numpy as np
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(81)
true_function = lambda x: np.sin(np.pi * x)

x_train = rng.uniform(-1.0, 1.0, 32)[:, None]
y_train = true_function(x_train[:, 0]) + rng.normal(scale=0.28, size=32)
x_validation = rng.uniform(-1.0, 1.0, 500)[:, None]
y_validation = true_function(x_validation[:, 0]) + rng.normal(scale=0.28, size=500)

polynomial = PolynomialFeatures(degree=12, include_bias=False)
scaler = StandardScaler()
train_features = scaler.fit_transform(polynomial.fit_transform(x_train))
validation_features = scaler.transform(polynomial.transform(x_validation))

# Add an unpenalized intercept column.
train_design = np.column_stack([np.ones(len(x_train)), train_features])
validation_design = np.column_stack([np.ones(len(x_validation)), validation_features])
weights = np.zeros(train_design.shape[1])

best_weights = weights.copy()
best_iteration = 0
best_validation_mse = np.inf
learning_rate = 0.015

for iteration in range(1, 5_001):
    residual = train_design @ weights - y_train
    gradient = (2.0 / len(y_train)) * train_design.T @ residual
    weights -= learning_rate * gradient

    validation_mse = np.mean((validation_design @ weights - y_validation) ** 2)
    if validation_mse < best_validation_mse:
        best_validation_mse = validation_mse
        best_iteration = iteration
        best_weights = weights.copy()

best_train_mse = np.mean((train_design @ best_weights - y_train) ** 2)
final_train_mse = np.mean((train_design @ weights - y_train) ** 2)
final_validation_mse = np.mean((validation_design @ weights - y_validation) ** 2)

print("best iteration:", best_iteration)
print("best train/validation MSE:", round(best_train_mse, 4), round(best_validation_mse, 4))
print("final train/validation MSE:", round(final_train_mse, 4), round(final_validation_mse, 4))
```

</details>

**Data augmentation** expands the observed sample with transformations believed to preserve the target: small image translations, audio perturbations, synonym-safe text transformations, or domain-specific simulations. It regularizes by encoding invariance. Its correctness depends on the label-preserving assumption. Horizontal reflection is sensible for many object images but wrong for text, asymmetric anatomy, or traffic signs. Augmentation changes the effective training distribution; it does not create independent information from nothing.

<details>
<summary><strong>Python example: test augmentation as an invariance assumption</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

plain_scores, augmented_scores = [], []

for seed in range(30):
    rng = np.random.default_rng(seed)
    x_train, y_train = make_moons(n_samples=40, noise=0.22, random_state=seed)
    x_test, y_test = make_moons(n_samples=3_000, noise=0.22, random_state=10_000 + seed)

    plain = DecisionTreeClassifier(random_state=seed)
    plain.fit(x_train, y_train)
    plain_scores.append(accuracy_score(y_test, plain.predict(x_test)))

    # Assume small coordinate perturbations preserve the moon label.
    copies = [x_train]
    labels = [y_train]
    for _ in range(8):
        copies.append(x_train + rng.normal(scale=0.06, size=x_train.shape))
        labels.append(y_train)

    augmented = DecisionTreeClassifier(random_state=seed)
    augmented.fit(np.vstack(copies), np.concatenate(labels))
    augmented_scores.append(accuracy_score(y_test, augmented.predict(x_test)))

print("mean plain accuracy:    ", round(np.mean(plain_scores), 4))
print("mean augmented accuracy:", round(np.mean(augmented_scores), 4))
print("mean difference:        ", round(np.mean(np.array(augmented_scores) - np.array(plain_scores)), 4))
```

</details>

The average over multiple samples is more informative than one favorable seed. Even then, the result supports only this synthetic invariance and model combination; stronger or semantically invalid perturbations can degrade generalization.


### **PAC Learning**

PAC learning turns "learnable" into a quantitative statement involving accuracy, confidence, sample size, and hypothesis-space complexity.

#### **Probably Approximately Correct Learning**

In a realizable binary-classification setting, a learner is **probably approximately correct** if, for every target concept in the class and every data distribution, sufficiently many examples allow it to return $\widehat h_S$ such that

$$
\Pr_{S\sim P^n}\left(R(\widehat h_S)\leq\varepsilon\right)\geq 1-\delta.
$$

$\varepsilon\in(0,1)$ is the tolerated population error and represents **approximately correct**. $\delta\in(0,1)$ is the maximum probability that random sampling leads to a worse model and represents **probably**. The probability is over the draw of the training sample, not over whether one already-trained model happens to classify a single point correctly.

![PAC learning separates the desired risk tolerance epsilon from the probability delta that a random sample produces an unacceptable model.](assets/pac-learning.svg){fig-align="center" width="100%" fig-alt="Repeated samples produce learned hypotheses; at least one minus delta have risk at most epsilon"}

*Diagram based on the PAC definitions in [Cornell CS6781, Theoretical Foundations of Machine Learning](https://www.cs.cornell.edu/courses/cs6781/2020sp/lectures/04-pac2.pdf).*

A sample-complexity function $m_{\mathcal H}(\varepsilon,\delta)$ specifies a sufficient $n$. A useful guarantee should grow polynomially in $1/\varepsilon$, $1/\delta$ or $\log(1/\delta)$, and a capacity measure of $\mathcal H$. PAC analysis is distribution-free in the sense that the guarantee holds for any fixed $P$ satisfying the setup, but it still assumes training and evaluation draws come from that same $P$.

PAC bounds are commonly conservative. Their value is not only predicting an exact required dataset size; they reveal which quantities fundamentally govern generalization and distinguish a guarantee from an empirical observation.

#### **Realizable and Agnostic Settings**

The **realizable** setting assumes that some $h\in\mathcal H$ has zero population error. Under noiseless threshold labels, for example, a threshold hypothesis can fit the target exactly. Realizability simplifies analysis but is strong: real labels can be noisy, features incomplete, and the true decision rule absent from $\mathcal H$.

The **agnostic** setting removes that assumption. The goal becomes excess risk relative to the best available hypothesis:

$$
R(\widehat h_S)
\leq \inf_{h\in\mathcal H}R(h)+\varepsilon
$$

with probability at least $1-\delta$. The unavoidable risk of the best member is not blamed on the learner. This formulation aligns more closely with practical supervised learning, where labels and representations rarely permit perfect prediction.

<details>
<summary><strong>Python example: contrast realizable and noisy threshold learning</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(93)

def fit_threshold_erm(x, y):
    # Candidate thresholds are the two outer sentinels and every midpoint.
    sorted_x = np.sort(np.unique(x))
    candidates = np.concatenate(
        [[sorted_x[0] - 1.0], (sorted_x[:-1] + sorted_x[1:]) / 2.0, [sorted_x[-1] + 1.0]]
    )
    errors = [np.mean((x >= threshold).astype(int) != y) for threshold in candidates]
    return candidates[int(np.argmin(errors))], min(errors)

true_threshold = 0.25
x_train = rng.uniform(-1.0, 1.0, 120)
clean_labels = (x_train >= true_threshold).astype(int)

# Agnostic case: labels are flipped independently with probability 0.15.
noisy_labels = clean_labels.copy()
flips = rng.random(len(noisy_labels)) < 0.15
noisy_labels[flips] = 1 - noisy_labels[flips]

x_population = rng.uniform(-1.0, 1.0, 200_000)
population_clean = (x_population >= true_threshold).astype(int)

for name, labels in [("realizable", clean_labels), ("agnostic/noisy", noisy_labels)]:
    threshold, train_error = fit_threshold_erm(x_train, labels)
    population_error_against_clean_rule = np.mean(
        (x_population >= threshold).astype(int) != population_clean
    )
    print(
        f"{name:15s} threshold={threshold:7.3f},"
        f" train error={train_error:.3f}, clean-rule population error={population_error_against_clean_rule:.3f}"
    )
```

</details>

Zero training error is expected in the realizable case once the sample brackets the target threshold. In the noisy case, no threshold can explain arbitrary flips, so the meaningful benchmark is the best threshold risk rather than zero.


### **VC Dimension and Shattering**

Finite hypothesis classes can be controlled by $\log|\mathcal H|$, but many important classes contain infinitely many parameter values. Vapnik-Chervonenkis theory measures the diversity of labelings a class can express on finite point sets.

A binary hypothesis class $\mathcal H$ **shatters** points $\{x_1,\ldots,x_m\}$ if, for every one of the $2^m$ possible binary label assignments, at least one $h\in\mathcal H$ realizes that assignment. Shattering does not ask whether one model fits every labeling simultaneously. It asks whether the class contains a potentially different model for each labeling.

The **VC dimension** is the largest $m$ for which at least one set of $m$ points can be shattered. If arbitrarily large sets can be shattered, the VC dimension is infinite.

![Affine linear separators in two dimensions can shatter three non-collinear points, but cannot realize the XOR labeling on four points.](assets/vc-shattering.svg){fig-align="center" width="100%" fig-alt="Eight labelings of three points separated by lines and a four-point XOR pattern that one line cannot separate"}

*Diagram based on [Cornell CS4780, Shattering and VC Dimension](https://www.cs.cornell.edu/courses/cs4780/2019fa/lectures/18-slt2.pdf).*

The figure establishes two facts for affine lines in $\mathbb R^2$: a suitable three-point set can be shattered, so the VC dimension is at least 3; no four-point set can be shattered, so it is at most 3. The second statement requires considering all possible arrangements, not only showing that one particular four-point set fails. The general result for affine linear classifiers in $\mathbb R^d$ is VC dimension $d+1$.

#### **Growth Functions**

The **growth function** counts the maximum number of distinct labelings that $\mathcal H$ can produce on $m$ points:

$$
\Pi_{\mathcal H}(m)
=\max_{x_1,\ldots,x_m}
\left|
\left\{(h(x_1),\ldots,h(x_m)):h\in\mathcal H\right\}
\right|.
$$

Always $\Pi_{\mathcal H}(m)\leq2^m$. The class shatters some $m$-point set exactly when $\Pi_{\mathcal H}(m)=2^m$. Once $m$ exceeds a finite VC dimension $d$, the Sauer-Shelah lemma bounds growth polynomially rather than exponentially:

$$
\Pi_{\mathcal H}(m)\leq\sum_{i=0}^{d}{m\choose i}
\leq\left(\frac{em}{d}\right)^d
\quad\text{for }m\geq d.
$$

This change from unrestricted exponential growth to capacity-controlled growth is what makes uniform convergence possible for many infinite classes.

<details>
<summary><strong>Python example: enumerate the growth function of thresholds on a line</strong></summary>

```python
import itertools
import numpy as np

def threshold_labelings(points):
    points = np.sort(np.asarray(points))
    candidates = np.concatenate(
        [[points[0] - 1.0], (points[:-1] + points[1:]) / 2.0, [points[-1] + 1.0]]
    )
    return {tuple((points >= threshold).astype(int)) for threshold in candidates}

for m in range(1, 7):
    points = np.arange(m, dtype=float)
    realized = threshold_labelings(points)
    all_binary = set(itertools.product([0, 1], repeat=m))
    print(
        f"m={m}: realized={len(realized):2d}, possible={len(all_binary):2d},"
        f" shattered={realized == all_binary}"
    )
```

</details>

One-dimensional thresholds produce only $m+1$ labelings: all zero, all one, and one transition at each gap. They shatter one point but not two, so their VC dimension is 1 under this orientation. Allowing both orientations changes the small-case count and VC dimension; the exact hypothesis definition matters.

#### **Model Complexity Beyond Parameter Count**

VC dimension is more informative than raw parameter count for many classical binary classes, but it is still a worst-case measure. Modern generalization may depend on quantities such as margins, weight norms, path norms, flatness or stability of the learned solution, compression, architecture-induced invariance, and the data distribution.

| Model family | Capacity controls more informative than a bare model name |
|---|---|
| linear classifier | input dimension, margin, feature norm, weight norm |
| decision tree | depth, leaves, minimum leaf size, pruning |
| kernel method | kernel, bandwidth, RKHS norm, margin |
| neural network | architecture, norms, margins, optimizer, augmentation, learned representation |

A model with many parameters can have low *effective* capacity on a particular distribution, while a smaller model can memorize through highly adaptive features. Capacity is therefore best treated as a property of the hypothesis-algorithm-data combination.


### **Concentration and Generalization Bounds**

Concentration inequalities formalize how a sample average approaches an expectation. The progression is important: first control one hypothesis, then control every hypothesis the learner might select, then solve the bound for the required sample size.

#### **Hoeffding's Inequality**

Let $Z_1,\ldots,Z_n$ be independent random variables in $[0,1]$ with common mean $\mu$. Hoeffding's inequality states

$$
\Pr\left(\left|\frac{1}{n}\sum_{i=1}^{n}Z_i-\mu\right|\geq\varepsilon\right)
\leq2\exp(-2n\varepsilon^2).
$$

For a **fixed** classifier $h$, take $Z_i=\mathbf1[h(x_i)\neq y_i]$. Then the sample mean is $\widehat R_S(h)$ and $\mu=R(h)$. The bound is distribution-free but requires independent observations and bounded loss. Squared loss is not automatically in $[0,1]`; it must be bounded, clipped with a changed objective, or handled with another inequality and tail assumption.

The exponential term shows that halving $\varepsilon$ requires roughly four times as many examples, while increasing confidence from $1-\delta$ affects sample size only logarithmically.

<details>
<summary><strong>Python example: compare empirical deviation probability with Hoeffding's bound</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(107)
event_probability = 0.30
epsilon = 0.10
trials = 30_000

print("n    empirical failure    Hoeffding upper bound")
for n in [20, 50, 100, 300, 800]:
    sample_means = rng.binomial(n=n, p=event_probability, size=trials) / n
    empirical_failure = np.mean(np.abs(sample_means - event_probability) >= epsilon)
    bound = min(1.0, 2.0 * np.exp(-2.0 * n * epsilon ** 2))
    print(f"{n:3d} {empirical_failure:18.6f} {bound:24.6f}")
```

</details>

The bound is an upper guarantee, not a prediction of the exact probability. It can be loose because it protects against every distribution satisfying the assumptions.

#### **Uniform Convergence**

Hoeffding controls one $h$ chosen independently of the sample. ERM chooses $h$ *after* inspecting the sample, so learning needs simultaneous control:

$$
\sup_{h\in\mathcal H}|R(h)-\widehat R_S(h)|\leq\varepsilon.
$$

For finite $\mathcal H$, apply Hoeffding to each hypothesis and the union bound:

$$
\Pr\left(
\sup_{h\in\mathcal H}|R(h)-\widehat R_S(h)|\geq\varepsilon
\right)
\leq2|\mathcal H|\exp(-2n\varepsilon^2).
$$

The logarithm of $|\mathcal H|$, rather than its raw size, appears after solving for $n$. For infinite classes, growth functions, VC dimension, Rademacher complexity, covering numbers, margins, or other capacity measures replace finite counting.

![Uniform convergence controls all candidate risks simultaneously, while stability controls how much the learned solution changes after replacing one example.](assets/uniform-convergence-stability.svg){fig-align="center" width="100%" fig-alt="Side-by-side diagrams of uniform convergence across hypotheses and stability under a one-example dataset replacement"}

Uniform convergence explains why ERM is safe when every empirical risk closely tracks its population counterpart. If the empirical minimizer is $\widehat h$ and each gap is at most $\varepsilon$, then

$$
R(\widehat h)
\leq \widehat R_S(\widehat h)+\varepsilon
\leq \widehat R_S(h^*_{\mathcal H})+\varepsilon
\leq R(h^*_{\mathcal H})+2\varepsilon.
$$

The middle inequality is exactly the ERM property. The two outer inequalities use uniform convergence for two data-dependent hypotheses.

<details>
<summary><strong>Python example: individual convergence is easier than uniform convergence</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(116)
thresholds = np.linspace(-0.9, 0.9, 31)
noise_rate = 0.10

# A large common population approximates the true risk of every threshold.
x_population = rng.uniform(-1.0, 1.0, 400_000)
clean_population = (x_population >= 0.05).astype(int)
population_labels = clean_population.copy()
flips = rng.random(len(population_labels)) < noise_rate
population_labels[flips] = 1 - population_labels[flips]
population_predictions = x_population[:, None] >= thresholds[None, :]
population_risks = np.mean(population_predictions != population_labels[:, None], axis=0)

epsilon = 0.15
n = 100
trials = 2_000
fixed_failures = 0
uniform_failures = 0

for _ in range(trials):
    x = rng.uniform(-1.0, 1.0, n)
    labels = (x >= 0.05).astype(int)
    flips = rng.random(n) < noise_rate
    labels[flips] = 1 - labels[flips]
    empirical_risks = np.mean((x[:, None] >= thresholds) != labels[:, None], axis=0)
    gaps = np.abs(empirical_risks - population_risks)
    fixed_failures += gaps[15] >= epsilon
    uniform_failures += gaps.max() >= epsilon

union_bound = min(1.0, 2 * len(thresholds) * np.exp(-2 * n * epsilon ** 2))
print("fixed-hypothesis failure:", round(fixed_failures / trials, 4))
print("uniform failure:        ", round(uniform_failures / trials, 4))
print("finite-class bound:     ", round(union_bound, 4))
```

</details>

The maximum over many candidates has more opportunities to find a sample-specific deviation. This multiple-comparisons effect is the theoretical analogue of extensive hyperparameter search over one validation set.

#### **Sample Complexity**

Setting the finite-class bound to at most $\delta$ and solving for $n$ gives the sufficient condition

$$
n\geq\frac{\log(2|\mathcal H|/\delta)}{2\varepsilon^2}.
$$

This bound guarantees that every empirical risk is within $\varepsilon$ of its population risk with confidence $1-\delta$. To guarantee ERM excess risk at most a requested value, constants may change because the earlier argument incurs $2\varepsilon$. Always check the theorem's exact event before quoting a sample-complexity formula.

<details>
<summary><strong>Python example: inspect how accuracy, confidence, and class size affect sample complexity</strong></summary>

```python
import math

def sufficient_samples(number_of_hypotheses, epsilon, delta):
    return math.ceil(math.log(2.0 * number_of_hypotheses / delta) / (2.0 * epsilon ** 2))

settings = [
    (100, 0.10, 0.05),
    (100, 0.05, 0.05),
    (1_000_000, 0.10, 0.05),
    (1_000_000, 0.10, 0.001),
]

print("|H|        epsilon   delta    sufficient n")
for size, epsilon, delta in settings:
    print(f"{size:9d} {epsilon:9.3f} {delta:7.3f} {sufficient_samples(size, epsilon, delta):13d}")
```

</details>

Three scaling laws are visible: $n$ grows as $1/\varepsilon^2$, logarithmically in $|\mathcal H|$, and logarithmically in $1/\delta$. The result is sufficient rather than necessary and ignores favorable distributional structure, margins, stability, and algorithm-specific bias, so real learning can require fewer or more useful examples depending on assumption violations.


### **Algorithmic Stability and Modern Generalization**

Classical capacity bounds ask whether all hypotheses in a class could generalize. Modern analyses often ask a narrower question: does the particular algorithm select solutions that are insensitive to individual observations? This shift matters when the nominal hypothesis class is enormous.

#### **Stability-Based Reasoning**

Let neighboring datasets $S$ and $S'$ differ in one example. A learning algorithm $A$ has **uniform stability** $\beta$ with respect to loss $\ell$ if, for every evaluation example $z$,

$$
|\ell(A(S),z)-\ell(A(S'),z)|\leq\beta.
$$

A small $\beta$ means no single observation can greatly change the learned predictor's loss. Stable algorithms tend to have small expected generalization gaps. Strong convexity and regularization can improve stability; highly adaptive memorization can weaken it. Stability is algorithm-specific: two optimizers searching the same architecture may have different behavior.

This reasoning does not require every member of a huge $\mathcal H$ to behave well. It only requires the mapping from dataset to selected model to be controlled. The exact theorem depends on boundedness, Lipschitz conditions, convexity, and the stability definition, so "stable" should not be used as an assumption-free synonym for robust.

*The stability concept in the preceding diagram follows [Bousquet and Elisseeff, Stability and Generalization](https://www.jmlr.org/papers/v2/bousquet02a.html).*

<details>
<summary><strong>Python example: regularization reduces sensitivity to replacing one example</strong></summary>

```python
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(127)
x = rng.uniform(-1.0, 1.0, 65)[:, None]
y = np.sin(np.pi * x[:, 0]) + rng.normal(scale=0.25, size=len(x))

# Neighboring dataset: replace one observation with an influential example.
x_neighbor = x.copy()
y_neighbor = y.copy()
x_neighbor[0, 0] = 0.98
y_neighbor[0] = 2.5

grid = np.linspace(-1.0, 1.0, 500)[:, None]

print("ridge alpha | RMS prediction change | maximum change")
for alpha in [1e-8, 0.01, 0.1, 1.0, 10.0]:
    def build_model():
        return make_pipeline(
            PolynomialFeatures(degree=12, include_bias=False),
            StandardScaler(),
            Ridge(alpha=alpha),
        )

    model_s = build_model().fit(x, y)
    model_neighbor = build_model().fit(x_neighbor, y_neighbor)
    change = model_s.predict(grid) - model_neighbor.predict(grid)
    root_mean_square_change = np.sqrt(np.mean(change ** 2))
    print(f"{alpha:11.8f} | {root_mean_square_change:21.5f} | {np.max(np.abs(change)):14.5f}")
```

</details>

The experiment measures prediction sensitivity, not the theorem's supremum over every possible neighboring dataset and evaluation point. It nevertheless makes the mechanism visible: stronger L2 shrinkage reduces how much one influential training observation can bend a high-degree fit.

#### **Double Descent**

The classical story predicts a U-shaped test-risk curve: as capacity increases, approximation improves until estimation error dominates. In many overparameterized systems, a second regime appears. Near the **interpolation threshold**, the model has just enough capacity to fit the training data exactly and solutions can be extremely sensitive. Far beyond that threshold, test risk may fall again as the algorithm selects a smoother, larger-margin, or minimum-norm interpolating solution. This pattern is called **double descent**.

![The classical U-shaped curve can be followed by a second descent after the interpolation threshold in some overparameterized learning systems.](assets/double-descent.svg){fig-align="center" width="100%" fig-alt="Classical U-shaped risk curve beside a modern double-descent curve with an interpolation threshold"}

*Redrawn from the conceptual Figure 1 in Belkin et al., [Reconciling Modern Machine-Learning Practice and the Classical Bias-Variance Trade-Off](https://pmc.ncbi.nlm.nih.gov/articles/PMC6689936/).*

Double descent does **not** prove that larger models always generalize better, nor does it invalidate bias-variance decomposition. It shows that a single monotonic notion of capacity is insufficient to predict bias and variance in modern parameterizations. The curve depends on the data distribution, noise, feature ordering, regularization, optimizer, architecture, and which interpolating solution is selected. Multiple-descent and monotone curves are also possible.

<details>
<summary><strong>Python example: observe double descent in minimum-norm linear regression</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(139)
n_train = 80
n_test = 700
max_features = 220
signal_features = 10
capacities = [5, 10, 20, 40, 60, 72, 78, 80, 82, 90, 110, 140, 180, 220]
repetitions = 10

risks = {capacity: [] for capacity in capacities}

for _ in range(repetitions):
    x_train_full = rng.normal(size=(n_train, max_features))
    x_test_full = rng.normal(size=(n_test, max_features))

    teacher = np.empty(max_features)
    teacher[:signal_features] = rng.normal(size=signal_features)
    # Later features carry weaker signal. Their ordering makes capacity explicit
    # while still allowing the far-overparameterized regime to recover structure.
    teacher[signal_features:] = 0.15 * rng.normal(size=max_features - signal_features)
    y_train = x_train_full @ teacher + rng.normal(scale=0.5, size=n_train)
    y_test = x_test_full @ teacher + rng.normal(scale=0.5, size=n_test)

    for capacity in capacities:
        x_train = x_train_full[:, :capacity]
        x_test = x_test_full[:, :capacity]

        # lstsq returns ordinary least squares for p < n and the
        # minimum-L2-norm interpolating solution for p > n.
        coefficients = np.linalg.lstsq(x_train, y_train, rcond=None)[0]
        mse = np.mean((x_test @ coefficients - y_test) ** 2)
        risks[capacity].append(mse)

print("features | median test MSE | interquartile range")
for capacity in capacities:
    values = np.asarray(risks[capacity])
    q25, q75 = np.quantile(values, [0.25, 0.75])
    print(f"{capacity:8d} | {np.median(values):15.4f} | [{q25:8.3f}, {q75:8.3f}]")
```

</details>

The peak near $p=n$ can be very large because an almost singular design matrix amplifies label noise. Past interpolation, the minimum-norm solution spreads fit across more directions and the additional weak signal features allow risk to decrease. This is a controlled Gaussian linear example; changing feature order, signal alignment, regularization, or noise can change or remove the pattern.


### **From Theory to Model Diagnosis**

Learning theory is most useful when it changes the next experiment. A disciplined diagnosis connects observed curves to several competing explanations instead of reflexively adding a larger model or more data.

| Observation | Plausible theoretical reading | Checks before acting | Candidate intervention |
|---|---|---|---|
| high train and validation error | approximation or optimization error | verify features, labels, convergence, baseline | richer representation, better optimizer, revised objective |
| low train error and large validation gap | estimation error or split mismatch | leakage audit, repeated splits, group/time structure | regularization, more representative data, lower effective capacity |
| gap shrinks with more data | variance-limited regime | confirm samples match deployment | collect or label more representative examples |
| both learning curves plateau poorly | bias, noisy target, missing information | Bayes/noise analysis, residual slices | better features, target definition, model family |
| validation good but deployment poor | $P_{train}\neq P_{deploy}$ or feedback | temporal, subgroup, drift, and causal checks | redesign split, adaptation, monitoring, retraining |
| highly variable result across seeds/splits | unstable estimation or optimization | confidence intervals and perturbation tests | stronger regularization, ensembling, more data |

A practical sequence is:

1. **Define the population and loss.** State what future examples are, what decision is made, and which errors matter. A bound for the wrong $P$ or $\ell$ is irrelevant.
2. **Audit the sampling assumptions.** Check independence, groups, time order, duplicates, leakage, and whether deployment is stationary relative to training.
3. **Protect the evaluation boundary.** Use training data for fitting, validation data for selection, and an untouched test set for final estimation. Report uncertainty, not only a point score.
4. **Plot validation and learning curves.** Separate capacity effects from sample-size effects and inspect variation across folds or seeds.
5. **Change one source of complexity at a time.** Vary model class, regularization, stopping time, or feature representation while preserving the evaluation protocol.
6. **Test stability and slices.** Replace or remove observations, perturb plausible inputs, and inspect subgroups. Average performance can hide a fragile model.
7. **Recheck after deployment.** Classical generalization guarantees do not protect against arbitrary distribution shift, strategic behavior, or feedback caused by the model itself.

<details>
<summary><strong>Python example: turn train-validation patterns into testable hypotheses</strong></summary>

```python
def diagnose(train_error, validation_error, baseline_error, tolerance=0.02):
    gap = validation_error - train_error
    near_baseline = validation_error >= baseline_error - tolerance

    if near_baseline and gap <= tolerance + 1e-12:
        return "Likely underfit or information-limited: inspect features, target noise, and optimization."
    if gap > tolerance:
        return "Likely estimation or split problem: audit leakage, capacity, regularization, and representativeness."
    if validation_error < baseline_error - tolerance:
        return "Promising validation result: quantify uncertainty and evaluate once on an untouched test set."
    return "Ambiguous pattern: repeat splits and inspect learning/validation curves."

cases = {
    "A": (0.31, 0.33, 0.35),
    "B": (0.04, 0.19, 0.35),
    "C": (0.08, 0.09, 0.35),
}

for name, values in cases.items():
    print(name, "->", diagnose(*values))
```

</details>

This helper is deliberately a hypothesis generator, not an automated verdict. Error scale, uncertainty, task costs, class imbalance, and distribution structure determine whether a numerical gap is meaningful.

The chapter's ideas form one chain:

$$
P \longrightarrow S \longrightarrow \mathcal H
\longrightarrow \widehat R_S
\longrightarrow \widehat h_S
\longrightarrow R(\widehat h_S).
$$

Population risk states the goal; empirical risk supplies a measurable proxy; inductive bias and capacity determine what can be selected; regularization changes the preference; PAC, VC, and concentration provide worst-case routes to guarantees; stability and modern interpolation explain why the selected algorithm can matter more than nominal parameter count. The next chapter moves from the statistical question "Will this rule generalize?" to the computational question "Which objective and optimization procedure produce the rule?"

[Back to Machine Learning guideline](Machine Learning.html)
